# Statystyka dla Data Analyst/Engineer — Moduł 1: Statystyka opisowa

Witaj w kursie statystyki! Jeśli przeszedłeś/aś nasze kursy Pythona i SQL, masz już
narzędzia do wyciągania danych i pisania kodu — teraz nauczymy się z tych danych
wnioskować. Statystyka dzieli się na dwie duże gałęzie: **opisową** (jak zwięźle
podsumować dane, które masz) i **wnioskującą** (jak na podstawie próbki wnioskować o
całej populacji — tym zajmiemy się od Modułu 5). Ten moduł to fundament: bez sprawnego
opisywania danych trudno je później testować.

## Spis treści
1. [Czym jest statystyka opisowa?](#sec1)
2. [Miary tendencji centralnej: średnia, mediana, dominanta](#sec2)
3. [Miary rozproszenia: wariancja i odchylenie standardowe](#sec3)
4. [Percentyle, kwartyle i IQR](#sec4)
5. [Wykrywanie obserwacji odstających](#sec5)
6. [`describe()` — szybkie podsumowanie w pandas](#sec6)
7. [Podsumowanie i ćwiczenia](#sec7)


Każdy moduł tego kursu zaczyna się od wygenerowania tego samego zestawu danych syntetycznych -- dzięki ustalonemu ziarnu losowości (`np.random.default_rng(42)`) liczby zawsze wyjdą identyczne, więc możesz otworzyć dowolny moduł niezależnie od pozostałych, nawet po restarcie kernela.

Przez cały ten moduł (i większość kolejnych) będziemy pracować na tym samym, syntetycznym zbiorze `klienci` — symulowanych danych sklepu internetowego. Kolumna `wydatki_miesieczne` będzie naszym głównym przykładem, bo ma realistyczny, prawoskośny kształt (większość klientów wydaje niewiele, garstka — bardzo dużo).

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 400

miasto = rng.choice(
    ["Warszawa", "Krakow", "Gdansk", "Wroclaw", "Poznan"],
    size=n, p=[0.30, 0.25, 0.15, 0.15, 0.15],
)
region = rng.choice(["Polnoc", "Poludnie", "Wschod", "Zachod"], size=n)
wiek = rng.integers(18, 70, size=n)

# wydatki miesieczne: rozklad prawoskosny (typowy dla danych o wydatkach)
wydatki_miesieczne = rng.lognormal(mean=5.2, sigma=0.5, size=n).round(2)

# satysfakcja skorelowana z wydatkami + efekt regionu (celowo rozny) + szum losowy,
# przyciete do skali 1-10 -- dzieki temu Modul 10 (ANOVA) ma realna roznice miedzy grupami
efekt_regionu = {"Polnoc": 0.0, "Poludnie": -0.5, "Wschod": 0.3, "Zachod": 0.8}
satysfakcja = np.clip(
    wydatki_miesieczne / 45
    + np.array([efekt_regionu[r] for r in region])
    + rng.normal(0, 1.5, size=n),
    1, 10,
).round(1)

# grupa kampanii A/B -- z celowo wprowadzonym efektem w konwersji (grupa B lepsza)
grupa = rng.choice(["A", "B"], size=n)
prawdopodobienstwo_konwersji = np.where(grupa == "A", 0.10, 0.20)
konwersja = rng.binomial(1, prawdopodobienstwo_konwersji)

# wydatki przed/po wprowadzeniu programu lojalnosciowego (dane sparowane)
wydatki_przed = rng.normal(220, 40, size=n).round(2)
efekt_programu = rng.normal(15, 10, size=n)
wydatki_po = (wydatki_przed + efekt_programu).round(2)

klienci = pd.DataFrame({
    "klient_id": np.arange(1, n + 1),
    "miasto": miasto,
    "region": region,
    "wiek": wiek,
    "wydatki_miesieczne": wydatki_miesieczne,
    "satysfakcja": satysfakcja,
    "grupa_kampanii": grupa,
    "konwersja": konwersja,
    "wydatki_przed": wydatki_przed,
    "wydatki_po": wydatki_po,
})
klienci.head()


<a id="sec1"></a>
## 1. Czym jest statystyka opisowa?

**Statystyka opisowa** odpowiada na pytanie: "jak zwięźle opisać zbiór liczb, żeby ktoś
inny (albo ty za tydzień) zrozumiał jego kształt, nie patrząc na wszystkie 400 wierszy?".
Zamiast wklejać całą kolumnę `wydatki_miesieczne`, mówimy np. "średnio klienci wydają
około 190 zł miesięcznie, ale rozrzut jest spory, od kilkudziesięciu do ponad 500 zł" —
to właśnie jest destylacja danych do kilku liczb, którymi zajmiemy się w tym module.

In [ ]:
klienci["wydatki_miesieczne"].describe()


Ta jedna linijka `pandas` już pokazuje większość tego, czego nauczymy się liczyć
ręcznie w tym module: liczbę obserwacji, średnią, odchylenie standardowe, minimum,
kwartyle i maksimum. Zaczniemy od rozłożenia tego na czynniki pierwsze.

<a id="sec2"></a>
## 2. Miary tendencji centralnej: średnia, mediana, dominanta

**Miary tendencji centralnej** próbują odpowiedzieć na pytanie: "jaka jest typowa,
centralna wartość w zbiorze?". Mamy trzy podstawowe:

- **Średnia (mean)** — suma wszystkich wartości podzielona przez ich liczbę. Wrażliwa na
  wartości odstające (outliers) — jedna bardzo duża wartość potrafi mocno ją podciągnąć.
- **Mediana (median)** — wartość środkowa po posortowaniu danych (albo średnia dwóch
  środkowych, gdy liczba obserwacji jest parzysta). Odporna na wartości odstające.
- **Dominanta (mode)** — najczęściej występująca wartość. Ma sens głównie dla danych
  dyskretnych/kategorycznych, rzadziej dla ciągłych pomiarów.

In [ ]:
import numpy as np
from scipy import stats

wydatki = klienci["wydatki_miesieczne"]

srednia = wydatki.mean()
mediana = wydatki.median()
dominanta = stats.mode(wydatki, keepdims=False)

print(f"Srednia: {srednia:.2f}")
print(f"Mediana: {mediana:.2f}")
print(f"Dominanta: {dominanta.mode:.2f} (wystapila {dominanta.count} razy)")


> ⚠️ **Średnia ≠ mediana, gdy rozkład jest skośny**
>
> Zauważ, że średnia wychodzi wyraźnie wyższa niż mediana. To sygnał, że rozkład jest **prawoskośny** (ma "ogon" w stronę dużych wartości) — kilku klientów wydających bardzo dużo podciąga średnią w górę, ale nie zmienia mediany. Zajmiemy się tym wizualnie w Module 2. Zasada praktyczna: gdy dane są skośne albo mają wartości odstające, **mediana lepiej opisuje "typowego" klienta** niż średnia.

> 📊 **Mediana zarobków, nie średnia zarobków**
>
> Dokładnie dlatego statystyki krajowe zwykle podają **medianę** wynagrodzeń, a nie średnią — średnia jest silnie podciągana przez niewielką grupę osób o bardzo wysokich zarobkach i nie odzwierciedla tego, ile zarabia "typowa" osoba.

<a id="sec3"></a>
## 3. Miary rozproszenia: wariancja i odchylenie standardowe

Sama miara centralna nie mówi nic o tym, jak bardzo dane są rozrzucone. Dwa zbiory mogą
mieć tę samą średnią, a wyglądać zupełnie inaczej — jeden skupiony blisko średniej, drugi
mocno rozrzucony. Do tego służą **miary rozproszenia**:

- **Rozstęp (range)** — różnica między maksimum a minimum. Prosty, ale bardzo wrażliwy na
  pojedyncze ekstremum.
- **Wariancja (variance)** — średnia kwadratów odchyleń od średniej. Jednostka jest
  podniesiona do kwadratu (np. "złote do kwadratu"), co utrudnia interpretację wprost.
- **Odchylenie standardowe (standard deviation)** — pierwiastek z wariancji. Wraca do
  oryginalnej jednostki (złote) i jest najczęściej używaną miarą rozproszenia.

In [ ]:
rozstep = wydatki.max() - wydatki.min()
wariancja = wydatki.var(ddof=1)
odchylenie_std = wydatki.std(ddof=1)

print(f"Rozstep: {rozstep:.2f}")
print(f"Wariancja: {wariancja:.2f}")
print(f"Odchylenie standardowe: {odchylenie_std:.2f}")


> ⚠️ **ddof=1: próbka kontra populacja**
>
> Argument `ddof` ("delta degrees of freedom") decyduje, czy liczysz wariancję dla **całej populacji** (`ddof=0`, dzielisz przez `n`) czy dla **próbki** reprezentującej większą populację (`ddof=1`, dzielisz przez `n-1`). Dzielenie przez `n-1` (tzw. poprawka Bessela) kompensuje fakt, że próbka z natury nieco *nie docenia* prawdziwego rozproszenia populacji. `pandas` domyślnie używa `ddof=1` -- w praktyce niemal zawsze pracujesz z próbką, więc to sensowny domyślny wybór.

<a id="sec4"></a>
## 4. Percentyle, kwartyle i IQR

**Percentyl** *p* to wartość, poniżej której leży *p*% obserwacji. Percentyl 50 to
dokładnie mediana. Szczególnie użyteczne są **kwartyle**: Q1 (percentyl 25), Q2 (percentyl
50 = mediana) i Q3 (percentyl 75) — dzielą dane na cztery równe pod względem liczności
części.

**Rozstęp międzykwartylowy (IQR — Interquartile Range)** to `Q3 - Q1`: rozpiętość
"środkowych" 50% danych, odporna na wartości odstające dużo bardziej niż zwykły rozstęp.

In [ ]:
q1 = wydatki.quantile(0.25)
q2 = wydatki.quantile(0.50)
q3 = wydatki.quantile(0.75)
iqr = q3 - q1

print(f"Q1 (25%): {q1:.2f}")
print(f"Q2 / mediana (50%): {q2:.2f}")
print(f"Q3 (75%): {q3:.2f}")
print(f"IQR: {iqr:.2f}")


`quantile()` przyjmuje dowolną wartość z przedziału 0-1, nie tylko kwartyle — np.
`wydatki.quantile(0.90)` da ci wartość, poniżej której leży 90% klientów (przydatne np.
do pytania "od jakiej kwoty zaczyna się top 10% klientów pod względem wydatków?").

<a id="sec5"></a>
## 5. Wykrywanie obserwacji odstających

**Obserwacja odstająca (outlier)** to wartość znacząco odbiegająca od reszty danych.
Najpopularniejsza, prosta metoda jej wykrywania opiera się właśnie na IQR z poprzedniej
sekcji: wszystko poniżej `Q1 - 1.5*IQR` albo powyżej `Q3 + 1.5*IQR` uznajemy za
potencjalny outlier. Mnożnik `1.5` to konwencja (dokładnie ta sama, której używa domyślny
wykres pudełkowy — zobaczysz go w Module 2), nie prawo natury.

In [ ]:
dolna_granica = q1 - 1.5 * iqr
gorna_granica = q3 + 1.5 * iqr

outliery = klienci[
    (klienci["wydatki_miesieczne"] < dolna_granica) | (klienci["wydatki_miesieczne"] > gorna_granica)
]
print(f"Granice: [{dolna_granica:.2f}, {gorna_granica:.2f}]")
print(f"Liczba outlierow: {len(outliery)}")
outliery[["klient_id", "wydatki_miesieczne"]].sort_values("wydatki_miesieczne", ascending=False)


> 📊 **Outlier to nie zawsze błąd**
>
> Łatwo założyć, że wartość odstająca to błąd w danych (literówka, awaria czujnika) — czasem tak jest, i wtedy trzeba ją poprawić albo usunąć. Ale czasem to po prostu prawdziwy, rzadki przypadek (klient VIP, który naprawdę wydaje dużo) i usunięcie go zniekształci obraz rzeczywistości. Reguła IQR mówi Ci **co sprawdzić ręcznie**, nie co automatycznie wyrzucić.

<a id="sec6"></a>
## 6. `describe()` — szybkie podsumowanie w pandas

Wróćmy do komórki z sekcji 1 — teraz już rozumiesz każdą liczbę, którą zwraca:

In [ ]:
klienci["wydatki_miesieczne"].describe()


`count` to liczba obserwacji, `mean`/`std` to średnia i odchylenie standardowe z
sekcji 2-3, `min`/`25%`/`50%`/`75%`/`max` to minimum, kwartyle i maksimum z sekcji 4.
Wywołanie `describe()` na całym DataFrame (`klienci.describe()`) policzy te same
statystyki dla wszystkich kolumn liczbowych naraz, a `klienci.describe(include="object")`
pokaże inny zestaw statystyk (m.in. najczęstszą wartość) dla kolumn tekstowych/
kategorycznych, takich jak `miasto`.

In [ ]:
klienci.describe(include="object")


<a id="sec7"></a>
## 7. Podsumowanie i ćwiczenia

W tym module poznaliśmy:
- miary tendencji centralnej: średnią, medianę, dominantę — i kiedy średnia myli,
- miary rozproszenia: rozstęp, wariancję, odchylenie standardowe (i różnicę `ddof=0` vs
  `ddof=1`),
- percentyle, kwartyle i IQR,
- prostą regułę IQR do wykrywania obserwacji odstających,
- `describe()` jako szybki, gotowy przegląd wszystkiego naraz.

> 📝 **Ćwiczenie 1: Statystyki wieku klientów**
>
> Policz średnią, medianę i odchylenie standardowe kolumny `wiek`. Czy średnia i mediana są bliskie siebie? Co to sugeruje o kształcie rozkładu wieku (w porównaniu do wydatków z sekcji 2)?

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
print(klienci["wiek"].mean())
print(klienci["wiek"].median())
print(klienci["wiek"].std(ddof=1))
```

Wiek jest generowany z rozkladu jednostajnego (rng.integers), wiec srednia i mediana powinny wyjsc bardzo blisko siebie -- w przeciwienstwie do wydatkow, wiek nie jest skosny.
</details>

> 📝 **Ćwiczenie 2: IQR i outliery dla satysfakcji**
>
> Policz Q1, Q3 i IQR dla kolumny `satysfakcja`, a następnie znajdź wiersze uznane za outliery według reguły `1.5 * IQR`. Ile ich jest?

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
q1_s = klienci["satysfakcja"].quantile(0.25)
q3_s = klienci["satysfakcja"].quantile(0.75)
iqr_s = q3_s - q1_s
dolna_s = q1_s - 1.5 * iqr_s
gorna_s = q3_s + 1.5 * iqr_s

outliery_s = klienci[
    (klienci["satysfakcja"] < dolna_s) | (klienci["satysfakcja"] > gorna_s)
]
len(outliery_s)
```
</details>

> 📝 **Ćwiczenie 3: Statystyki wg miasta**
>
> Użyj `groupby("miasto")["wydatki_miesieczne"]` razem z `.agg([...])` (poznanym w kursie Pythona/pandas), żeby policzyć średnią, medianę i odchylenie standardowe wydatków osobno dla każdego miasta.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
klienci.groupby("miasto")["wydatki_miesieczne"].agg(["mean", "median", "std"])
```
</details>

> 📝 **Ćwiczenie 4: Top 10% klientów**
>
> Znajdź próg wydatków, powyżej którego znajduje się top 10% klientów (podpowiedź: `quantile(0.90)`), a następnie zwróć DataFrame z tymi klientami, posortowany malejąco po wydatkach.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
prog_top10 = klienci["wydatki_miesieczne"].quantile(0.90)
top_klienci = klienci[klienci["wydatki_miesieczne"] > prog_top10]
top_klienci.sort_values("wydatki_miesieczne", ascending=False)
```
</details>

> 🔥 **Wyzwanie: własna funkcja podsumowująca**
>
> Napisz funkcję `podsumuj(kolumna)`, która przyjmuje Series liczbowy i zwraca słownik z kluczami: `srednia`, `mediana`, `odchylenie_std`, `iqr`, `liczba_outlierow` (policzoną regułą z sekcji 5). Przetestuj ją na `wydatki_miesieczne` i na `wiek`, porównując wyniki.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
def podsumuj(kolumna):
    q1 = kolumna.quantile(0.25)
    q3 = kolumna.quantile(0.75)
    iqr = q3 - q1
    dolna = q1 - 1.5 * iqr
    gorna = q3 + 1.5 * iqr
    liczba_outlierow = ((kolumna < dolna) | (kolumna > gorna)).sum()
    return {
        "srednia": kolumna.mean(),
        "mediana": kolumna.median(),
        "odchylenie_std": kolumna.std(ddof=1),
        "iqr": iqr,
        "liczba_outlierow": liczba_outlierow,
    }


print(podsumuj(klienci["wydatki_miesieczne"]))
print(podsumuj(klienci["wiek"]))
```
</details>

## Co dalej?

W **Module 2** przejdziemy od liczb do obrazków: histogramy, wykresy pudełkowe (boxplot) i
wykresy gęstości pokażą dokładnie to, o czym już wiesz z tego modułu (skośność, IQR,
outliery), ale w formie, którą można "zobaczyć jednym rzutem oka" zamiast czytać kolumnę
liczb.